# Week 29, Day 3: Operator overloading

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w029d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
print(2 + 3, "ab" + "cd", [1] + [2], (1,) * 3)
print((5).__add__(3), "ab".__add__("cd"), int.__mul__(4, 5))
try:
    print("ab" + 3)
except TypeError as e:
    print("TypeError:", e)

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"

    def __eq__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return (self.x, self.y) == (other.x, other.y)

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __sub__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x - other.x, self.y - other.y)

    def __mul__(self, k):
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vector(self.x * k, self.y * k)

a, b = Vector(1, 2), Vector(3, -1)
print(a + b, a - b, a * 3, a + b == Vector(4, 1))
total = a
total += b          # no __iadd__, so this is total = total + b: a new object
print(total, a)
try:
    a + 5
except TypeError as e:
    print("TypeError:", e)

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"

    def __add__(self, other):
        if not isinstance(other, Vector):
            return NotImplemented
        return Vector(self.x + other.x, self.y + other.y)

    def __radd__(self, other):
        if other == 0:              # lets sum() start from 0
            return self
        return self.__add__(other)

    def __mul__(self, k):
        if not isinstance(k, (int, float)):
            return NotImplemented
        return Vector(self.x * k, self.y * k)

    __rmul__ = __mul__              # k * v means the same as v * k

v = Vector(1, 2)
print(3 * v, v * 3)
print(sum([Vector(1, 1), Vector(2, 0), Vector(0, 5)]))

In [ ]:
import math

class Vector:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vector({self.x!r}, {self.y!r})"

    def __neg__(self):
        return Vector(-self.x, -self.y)

    def __abs__(self):
        return math.hypot(self.x, self.y)

    def __bool__(self):
        return self.x != 0 or self.y != 0

v = Vector(3, 4)
print(-v, abs(v), bool(v), bool(Vector(0, 0)))
for w in [v, Vector(0, 0)]:
    print("moving" if w else "still")

> **Key idea.** Operators call special methods on the left operand: `__add__`, `__sub__`, `__mul__`, `__truediv__`, `__neg__`, `__abs__`. Return a new object of the right type, and `NotImplemented` for types you don't support. Reflected methods (`__radd__`, `__rmul__`) handle `3 * v` and `sum()`. `__bool__` decides truth. Overload only where the meaning is obvious.

### Exercise: Money that adds up

Give `Money` (whole pence) these operators: `+` and `-` with another `Money`; `*` with a whole number, on either side (`Money(250) * 3` and `3 * Money(250)`); `-m` and `abs(m)`; `bool(m)` (false only for zero); and `sum()` support. `Money + 5`, `Money * Money` and `Money * 1.5` must raise `TypeError`. Every result is a new `Money`.

In [ ]:
class Money:
    def __init__(self, pence):
        self.pence = pence

    def __repr__(self):
        return f"Money({self.pence})"

    def __eq__(self, other):
        if not isinstance(other, Money):
            return NotImplemented
        return self.pence == other.pence

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_ops():\n    \"operators\"\n    M = ns[\"Money\"]\n    a, b = M(250), M(100)\n    assert a + b == M(350) and a - b == M(150) and a * 3 == M(750) and 3 * a == M(750)\n    assert -a == M(-250) and abs(M(-40)) == M(40) and bool(M(0)) is False and bool(M(1)) is True\n    assert sum([M(1), M(2), M(3)]) == M(6) and sum([], M(0)) == M(0)\n    assert a == M(250) and b == M(100), \"operators must not change their operands\"\n    assert isinstance(a + b, M) and isinstance(-a, M)\ndef test_refused():\n    \"nonsense refused\"\n    M = ns[\"Money\"]\n    for f in (lambda: M(1) + 5, lambda: M(1) * M(2), lambda: M(1) * 1.5, lambda: 5 - M(1), lambda: M(1) - 5):\n        try:\n            f()\n            assert False, \"this should raise TypeError\"\n        except TypeError:\n            pass")

### Exercise: Durations

Write `Duration(minutes)` (a whole number, 0 or more, or `ValueError`), with `__repr__` like `Duration(95)` and `__str__` like `1h 35m` (or `35m` under an hour, `2h` for whole hours, `0m` for zero). Support `+` between durations, `-` (raising `ValueError` if the result would be negative), `*` and `/` by a whole number (the division rounds down: `Duration(95) / 2` is `Duration(47)`), and `==`.

In [ ]:
class Duration:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_text():\n    \"text\"\n    D = ns[\"Duration\"]\n    assert [str(D(m)) for m in (95, 35, 120, 0, 61)] == [\"1h 35m\", \"35m\", \"2h\", \"0m\", \"1h 1m\"] and repr(D(95)) == \"Duration(95)\"\ndef test_ops():\n    \"operators\"\n    D = ns[\"Duration\"]\n    assert D(30) + D(45) == D(75) and D(90) - D(30) == D(60) and D(20) * 3 == D(60) and 3 * D(20) == D(60)\n    assert D(95) / 2 == D(47) and D(10) != D(11)\n    for f, err in [(lambda: D(10) - D(20), ValueError), (lambda: D(-1), ValueError), (lambda: D(5) + 5, TypeError), (lambda: D(5) / D(5), TypeError)]:\n        try:\n            f()\n            assert False, f\"should raise {err.__name__}\"\n        except err:\n            pass")

### Exercise: Order matters

Write `__sub__` for `Temperature` so that `t - number` gives a new `Temperature` that many degrees colder, and `t1 - t2` gives the **difference as a plain number** of degrees. Subtracting a temperature from a number (`100 - t`) makes no sense, so it must raise `TypeError`, as must subtracting anything else.

In [ ]:
class Temperature:
    def __init__(self, celsius):
        self.celsius = celsius

    def __repr__(self):
        return f"Temperature({self.celsius!r})"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"subtraction\"\n    T = ns[\"Temperature\"]\n    cooler = T(21.5) - 3\n    assert isinstance(cooler, T) and cooler.celsius == 18.5\n    assert T(21.5) - T(18) == 3.5 and type(T(20) - T(5)) in (int, float)\n    try:\n        100 - T(5)\n        assert False, \"100 - Temperature should raise TypeError\"\n    except TypeError:\n        pass\n    try:\n        T(5) - \"3\"\n        assert False, \"subtracting text should raise TypeError\"\n    except TypeError:\n        pass")

### Bug hunt: Adding changed it

After `c = a + b`, `a` has changed too. Fix `__add__` so it leaves both operands alone.

In [ ]:
class Basket:
    def __init__(self, items=None):
        self.items = list(items or [])

    def __repr__(self):
        return f"Basket({self.items!r})"

    def __add__(self, other):
        if not isinstance(other, Basket):
            return NotImplemented
        self.items.extend(other.items)
        return self

a = Basket(["tea"])
b = Basket(["milk"])
c = a + b
print(a, b, c)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"operands unchanged\"\n    expect_output(\"Basket(['tea']) Basket(['milk']) Basket(['tea', 'milk'])\")")

### Bug hunt: Works one way only

`v * 2` works but `2 * v` crashes with `TypeError: unsupported operand type(s) for *: 'int' and 'Vec'`, and `sum` of vectors crashes too. Fix the class (two additions).

In [ ]:
class Vec:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def __repr__(self):
        return f"Vec({self.x}, {self.y})"

    def __add__(self, other):
        if not isinstance(other, Vec):
            return NotImplemented
        return Vec(self.x + other.x, self.y + other.y)

    def __mul__(self, k):
        return Vec(self.x * k, self.y * k)

v = Vec(1, 2)
print(v * 2, 2 * v, sum([v, v, v]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"both ways\"\n    expect_output(\"Vec(2, 4) Vec(2, 4) Vec(3, 6)\")")

### Bug hunt: A confusing error

`Score(5) + "five"` should raise the usual `TypeError: unsupported operand type(s)`. Instead it crashes inside the method with a confusing `AttributeError`. Fix `__add__`.

In [ ]:
class Score:
    def __init__(self, points):
        self.points = points

    def __add__(self, other):
        return Score(self.points + other.points)

try:
    Score(5) + "five"
except TypeError as e:
    print("TypeError:", e)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"a proper TypeError\"\n    expect_output(\"TypeError: unsupported operand type(s) for +: 'Score' and 'str'\")")

### Exercise: Predict them



In [ ]:
class N:
    def __init__(self, v):
        self.v = v
    def __add__(self, o):
        return N(self.v + (o.v if isinstance(o, N) else o))
    def __radd__(self, o):
        return N(o * 10 + self.v)
    def __bool__(self):
        return self.v > 0

p = ___   # (N(1) + N(2)).v
q = ___   # (N(1) + 5).v
r = ___   # (5 + N(1)).v
s = ___   # bool(N(0))
t = ___   # (sum([N(1), N(2)])).v    (sum starts with 0)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r, s, t) == (3, 6, 51, False, 3)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: polynomials

Write `Poly(coefficients)` for polynomials with whole-number coefficients, given lowest power first: `Poly([1, 0, 3])` is 3x² + 1. Store them without trailing zeros (`Poly([2, 0, 0])` is the same as `Poly([2])`; the zero polynomial has an empty list). Support:

- `==` and hashing (by coefficients);
- `+` and `-` between polynomials, and with whole numbers on either side (a number is a constant polynomial: `5 - p` works);
- `*` between polynomials, and with whole numbers on either side;
- `-p`, and `bool(p)` (false only for zero);
- `__repr__` giving `Poly([1, 0, 3])`, and `__str__` giving `3x^2 + 1`, highest power first, skipping zero terms, writing `x` not `1x` and `-x` not `-1x`, `x` not `x^1`, and `-` joins for negative terms: `Poly([-1, 1, -2])` is `-2x^2 + x - 1`. Zero is `0`.

In [ ]:
class Poly:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_basic():\n    \"storage, equality, repr\"\n    P = ns[\"Poly\"]\n    assert P([2, 0, 0]) == P([2]) and repr(P([1, 0, 3, 0])) == \"Poly([1, 0, 3])\" and P([]) == P([0, 0])\n    assert len({P([1, 2]), P([1, 2, 0]), P([2, 1])}) == 2 and not P([0]) and P([0, 1])\ndef test_arith():\n    \"arithmetic\"\n    P = ns[\"Poly\"]\n    p, q = P([1, 1]), P([-1, 1])\n    assert p * q == P([-1, 0, 1]) and p + q == P([0, 2]) and p - p == P([])\n    assert 2 * p == P([2, 2]) and p * 2 == P([2, 2]) and p + 3 == P([4, 1]) and 3 + p == P([4, 1])\n    assert 5 - p == P([4, -1]) and p - 5 == P([-4, 1]) and -p == P([-1, -1])\n    assert p == P([1, 1]), \"operands must not change\"\n    try:\n        p + \"x\"\n        assert False, \"adding text should raise TypeError\"\n    except TypeError:\n        pass\ndef test_str():\n    \"str\"\n    P = ns[\"Poly\"]\n    cases = {(1, 0, 3): \"3x^2 + 1\", (-1, 1, -2): \"-2x^2 + x - 1\", (): \"0\", (0, -1): \"-x\", (5,): \"5\", (0, 0, 1): \"x^2\", (-4,): \"-4\", (0, 2, 0, -1): \"-x^3 + 2x\"}\n    for coeffs, want in cases.items():\n        assert str(P(list(coeffs))) == want, (coeffs, str(P(list(coeffs))))")

## Warm-up

`2 + 3` is really `(2).__add__(3)`: every operator is a method call on its left operand. Strings have their own `__add__`, lists theirs, and a string's refuses a number. Your classes can define these methods too.

## Operators are method calls

| Expression | Method called |
|---|---|
| `a + b` | `a.__add__(b)` |
| `a - b` | `a.__sub__(b)` |
| `a * b` | `a.__mul__(b)` |
| `a / b` | `a.__truediv__(b)` |
| `a // b` | `a.__floordiv__(b)` |
| `a % b` | `a.__mod__(b)` |
| `-a` | `a.__neg__()` |
| `abs(a)` | `a.__abs__()` |

Three rules, all visible here:

1. **Return a new object**; don't change `self` or `other`. Nobody expects `a + b` to change `a`. (`total += b` then works by making a new vector and rebinding `total`, leaving `a` alone. A class can define `__iadd__` to change itself in place, as lists do, but for value-like objects, don't.)
2. **Return `NotImplemented`** for types the operator doesn't support. Python then tries the other operand (next section), and if that fails too, raises a clear `TypeError` naming both types.
3. **Return the right type**, usually the same class: vector plus vector is a vector.

## The other way round

`a * 3` calls `a.__mul__(3)`. But `3 * a` calls `(3).__mul__(a)` first, and `int` knows nothing about vectors, so it returns `NotImplemented`. Python then tries the **reflected** method on the right operand: `a.__rmul__(3)`.

- Every arithmetic operator has a reflected twin: `__radd__`, `__rsub__`, `__rmul__`, `__rtruediv__` and so on. It's called only when the left operand's method returned `NotImplemented` (or doesn't exist).
- `__rmul__ = __mul__` just gives the same method a second name, which is fine when the order doesn't matter. For `-` and `/` it does matter: `__rsub__(self, other)` must work out `other - self`.
- `sum(items)` starts from `0` and does `0 + first`. Supporting that needs `__radd__` to accept `0`.

## Unary operators and truth

`-v` calls `__neg__`, `+v` calls `__pos__`, and `abs(v)` calls `__abs__`. `bool(v)`, and so `if v:`, calls `__bool__`; without it Python tries `__len__` (Week 31), and otherwise every object is true.

Make `__bool__` mean something obvious: "is it empty?", "is it zero?". If it isn't obvious what True should mean, don't define it.

## When to overload, and when not

Operator overloading is brilliant when the operator means **what everyone would expect**: adding money, scaling vectors, combining times, `-` for "the opposite". Python's own types and well-loved libraries do it (`datetime - datetime` gives a `timedelta`; `pathlib.Path("a") / "b"` joins paths).

It's harmful when it's clever: `basket + item` that changes the basket, `user * 2` meaning anything at all, `-account` closing it. A method with a name (`basket.add(item)`) is clearer than an operator whose meaning has to be learned.

And support only what makes sense. Money plus money is money; money times a number is money; but money times money, or money plus a plain number (pounds? pence?), should be a `TypeError`, which `NotImplemented` gives you for free.

## Practice

## Quiz

## Daily challenge

## Before you go

Look for a class you've written where an operator would read naturally (adding two of something, scaling one by a number), and one where it wouldn't. Tomorrow: properties, which make attributes that check what they're given and work out values on the fly.